# Financial Consumer Complaints NLP Classification (Leakage-Free)

This notebook reproduces the full exploratory analysis, leakage audit, taxonomy consolidation, multi-model TF-IDF benchmarking, and feature interpretability workflow on the public **CFPB Consumer Complaint Database**.

### Key Methodological Safeguards
1. **No Target Leakage**: Uses **only** `Consumer complaint narrative` to predict `Product` (excluding `Sub-product`, `Issue`, and `Sub-issue`).
2. **No Data Leakage**: Stratified 70/15/15 Train/Validation/Test split is performed before any vectorization; `ComplaintTextCleaner` and `TfidfVectorizer` live inside `sklearn.pipeline.Pipeline`.
3. **No Duplicate Template Leakage**: Exact normalized narratives (after stripping CFPB `XXXX` and `XX/XX/YYYY` redaction masks) are deduplicated prior to splitting, and exact texts with conflicting product labels are dropped.
4. **Historical Taxonomy Consolidation**: Consolidates pre-April 2017 legacy CFPB labels into 9 canonical financial product categories (eliminating temporal label drift).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from complaints_nlp.config import Settings
from complaints_nlp.modeling import (
    predict_complaints,
    save_evaluation_plots,
    save_model_bundle,
    train_and_evaluate,
)
from complaints_nlp.preprocessing import load_complaints_csv

settings = Settings.from_env()
print("Dataset path:", settings.data_path)
print("Model output:", settings.model_path)

## 1. Load, Consolidate Taxonomy (18 → 9 Classes), and Deduplicate Normalized Narratives

Download `rows.csv` from [Kaggle (CFPB Consumer Complaint Database)](https://www.kaggle.com/datasets/selener/consumer-complaint-database/data) or the official [CFPB Open Data portal](https://www.consumerfinance.gov/data-research/consumer-complaints/) and place it in `data/rows.csv` (or configure `DATA_PATH` in `.env`).

In [ ]:
# Set MAX_ROWS = 50_000 for quick local iteration, or None for the full deduplicated dataset (~359k rows).
MAX_ROWS = None

df, stats = load_complaints_csv(
    csv_path=settings.data_path,
    consolidate=True,
    drop_deprecated=True,
    deduplicate=True,
    max_rows=MAX_ROWS,
    random_state=42,
)

print(pd.Series(stats.to_dict()).drop("class_counts"))
print("\nCanonical class distribution:")
print(pd.Series(stats.class_counts))

## 2. Inspect Cleaned Narratives (CFPB `XXXX` Masks & Redacted Dates Removed)

In [ ]:
for idx in range(min(3, len(df))):
    row = df.iloc[idx]
    print(f"--- Example {idx + 1} | Product: {row['product']} ---")
    print("RAW    :", row["raw_text"][:220].replace("\n", " "))
    print("CLEANED:", row["cleaned_text"][:220])
    print()

## 3. Stratified 70/15/15 Split, 6-Model Validation Benchmark & Held-Out Test Evaluation

Benchmarks `MultinomialNB`, `ComplementNB`, `LogisticRegression` (default & balanced), and `LinearSVC` (default & balanced) inside leak-free `Pipeline`s. The winner is selected strictly on **Validation Macro-F1** (with train-only CV probability calibration if a `LinearSVC` variant wins) and evaluated **once** on the held-out Test split.

In [ ]:
best_pipeline, report, plot_payload = train_and_evaluate(
    texts=df["cleaned_text"].tolist(),
    labels=df["product"].tolist(),
    dataset_stats=stats,
    max_features=50_000,
    min_df=5,
    random_state=42,
)

bench_df = pd.DataFrame(report["validation_benchmarks"]).sort_values("val_macro_f1", ascending=False)
print("=== Validation Split Benchmark ===")
display(bench_df)

print("\n=== Held-Out Test Metrics (Selected Model) ===")
print(report["selected_model"])
print(report["test_metrics"])

per_class_df = pd.DataFrame(report["test_per_class"]).T.sort_values("f1", ascending=False)
display(per_class_df)

## 4. Save Model Artifact & Diagnostic Plots

In [ ]:
save_model_bundle(best_pipeline, metadata=report, output_path=settings.model_path)
saved_plots = save_evaluation_plots(PROJECT_ROOT / "artifacts" / "plots", report, plot_payload)
print("Saved model:", settings.model_path)
print("Saved plots:", [p.name for p in saved_plots])

## 5. Sanity-Check Inference on Unseen Complaint Text

In [ ]:
sample_texts = [
    "On XX/XX/2018 my mortgage servicer misapplied my escrow payment and charged late fees despite timely autopay.",
    "A debt collector keeps calling my workplace about an old medical bill that I already disputed in writing.",
    "My credit card statement shows an unauthorized charge of {$420.00} and the issuer refused my dispute.",
]
for text, pred in zip(sample_texts, predict_complaints(best_pipeline, sample_texts, top_k=3)):
    print(f"Text : {text}")
    print(f"Pred : {pred['predicted_product']} ({pred['confidence']:.1%})")
    print(f"Top-3: {pred['top_predictions']}\n")